# Bulk RNA-seq counts normalization



## Description





The normalization step follows steps used by the GTeX pipeline. Genes are first filtered to keep genes where TPM is greater than 10% in at least 20% of the samples. They are also kept if read counts is greater than 6 in at least 20% of the samples. The filtered data is then normalized using the Trimmed Mean of M-value (TMM) method. 



## Input

1. TPM matrix and read count matrix in RNA-SeQC format
    - the first two rows should be commented text with `#` prefix.
    - the matrix should be tab delimited.
    - the matrix files should end with `gct` suffix
    - These requirements are satisfied if the inputs are outputs from [`bulk_expression_QC` pipeline](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/QC/bulk_expression_QC.html).
2. GTF for collapsed gene model
    - the gene names must be consistent with the GCT matrices (eg ENSG00000000003 vs. ENSG00000000003.1 will not work) 
    - chromosome names must have `chr` prefix (although we can make it an option in the pipeline, currently we assume the `chr` prefix convention)
3. Meta-data to match between sample names in expression data and genotype files
    - Required input
    - Tab delimited with header
    - Only 2 columns: first column is sample name in expression data, 2nd column is sample name in genotype data
    - **must contains all the sample name in expression matrices even if they don't existing in genotype data**

## Output

Normalized expression file in `bed` format.

## Minimal Working Example Steps

These commands assume the MWE data bundle is available as `mwe_data/` in the repository root. Run each command from the repository root; commands that reference `output/mwe_notebook/` expect the upstream MWE commands in this section to have produced those files.


### Multi-sample read-count normalization

This step consumes the filtered TPM and count GCT files generated by `bulk_expression_QC.ipynb qc`.


In [ ]:
sos run pipeline/bulk_expression_normalization.ipynb normalize \
    --cwd output/mwe_notebook/rnaseq \
    --counts-gct output/mwe_notebook/rnaseq/AC_bam_list.normalized.rnaseqc.low_expression_filtered.outlier_removed.geneCount.gct.gz \
    --tpm-gct output/mwe_notebook/rnaseq/AC_bam_list.normalized.rnaseqc.low_expression_filtered.outlier_removed.tpm.gct.gz \
    --annotation-gtf mwe_data/rnaseqc_compatible.gtf \
    --sample_participant_lookup mwe_data/sample_participant_lookup.tsv \
    --count-threshold 0 \
    --sample-frac-threshold 0 \
    --normalization-method tmm_cpm_voom \
    --modular-script-dir code/script


## Troubleshooting

| Step | Substep | Problem | Possible Reason | Solution |
|------|---------|---------|------------------|---------|
|  |  |  |  |  |




## Command interface

In [4]:
!sos run bulk_expression_normalization.ipynb -h

/home/caox2/.pixi/envs/python/lib/python3.12/site-packages/sos/targets.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
usage: sos run bulk_expression_normalization.ipynb
               [workflow_name | -t targets] [options] [workflow_options]
  workflow_name:        Single or combined workflows defined in this script
  targets:              One or more targets to generate
  options:              Single-hyphen sos parameters (see "sos run -h" for details)
  workflow_options:     Double-hyphen workflow-specific parameters

Workflows:
  normalize

Global Workflow Options:
  --cwd output (as path)
                        Work directory & output directory
  --counts-gct VAL (as path, required)
                        gene count table
  --tpm-gct VAL (as path, required)
 

In [9]:
[global]
parameter: modular_script_dir = path('code/script')  # override with --modular-script-dir
# Work directory & output directory
parameter: cwd = path("output")
#  gene count table
parameter: counts_gct = path
#  gene TPM table
parameter: tpm_gct = path
#  gene gtf annotation table
parameter: annotation_gtf = path
# A file to map sample ID from expression to genotype,must contain two columns, sample_id and participant_id, mapping IDs in the expression files to IDs in the genotype (these can be the same).
parameter: sample_participant_lookup = path
parameter: tpm_threshold = 0.1
parameter: count_threshold = 6
parameter: sample_frac_threshold = 0.2
# Normalization method: TMM + CPM(voom) (tmm_cpm_voom), TMM + CPM(edgeR) (tmm_cpm_edger), or quantile normalization (qn)
parameter: normalization_method = 'tmm_cpm_voom'
# Quantile Normalization after rescale
parameter: quantile_normalize = True
# For cluster jobs, number commands to run per job
parameter: job_size = 1
# Wall clock time expected
parameter: walltime = "5h"
# Memory expected
parameter: mem = "16G"
# Number of threads
parameter: numThreads = 20
parameter: container = ""
import re
parameter: entrypoint= ('micromamba run -a "" -n' + ' ' + re.sub(r'(_apptainer:latest|_docker:latest|\.sif)$', '', container.split('/')[-1])) if container else ""

In [ ]:
[normalize]
# Path to the input molecular phenotype data, should be a processd and indexed bed.gz file, with tabix index.
input: tpm_gct, counts_gct, annotation_gtf, sample_participant_lookup
output: f'{cwd:a}/{_input[0]:bnnn}.{normalization_method}.{"no_qnorm." if not quantile_normalize else ""}expression.bed.gz'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output[0]:bn}'  
bash: expand= "${ }", stderr = f'{_output[0]:nn}.stderr', stdout = f'{_output[0]:nn}.stdout', container = container, entrypoint = entrypoint
    python3 ${modular_script_dir}/molecular_phenotypes/QC/bulk_expression_normalization.py \
        --step normalize \
        --cwd "${cwd}" \
        --tpm-gct "${_input[0]}" \
        --counts-gct "${_input[1]}" \
        --annotation-gtf "${_input[2]}" \
        --sample-participant-lookup "${_input[3]}" \
        --tpm-threshold ${tpm_threshold} \
        --count-threshold ${count_threshold} \
        --sample-frac-threshold ${sample_frac_threshold} \
        --normalization-method "${normalization_method}" \
        ${"--quantile-normalize" if quantile_normalize else ""} \
        --numThreads ${numThreads}
